# Лабораторная работа №5
## Выполнил: Лавренов М.А.
## Группа: ИУ5-23М



###**Цель лабораторной работы:**
Ознакомление с базовыми методами обучения с подкреплением.

###**Задание:**
<ol>
<li>На основе рассмотренного на лекции примера реализуйте алгоритм Policy Iteration для любой среды обучения с подкреплением (кроме рассмотренной на лекции среды Toy Text / Frozen Lake) из библиотеки Gym (или аналогичной библиотеки).
</li>

###**Ход выполнения задания**

---



## Установка зависимостей

Установка необходимых библиотек и зависимостей для работы с Atari-играми

In [ ]:
# Fix for Python 3.12 compatibility issues
!pip install --upgrade pip setuptools wheel

# Standard setup for rendering
!apt-get update > /dev/null 2>&1
!apt-get install -y xvfb python-opengl ffmpeg cmake > /dev/null 2>&1

# Specific versions to avoid ImpImporter error in Python 3.12
!pip install "setuptools<68" "moviepy" pyvirtualdisplay gymnasium[toy-text] pygame > /dev/null 2>&1

^C
^C


Установка библиотек, необходимые для работы с графикой и видео

In [ ]:
!apt-get install xvfb
!apt-get install python3-opengl ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 60 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-opengl is already the newest version (3.1.5+dfsg-1).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 60 not upgraded.


Создать папку с названием "video"

In [ ]:
!mkdir -p video

## Импорт библиотек

In [ ]:
import gymnasium as gym

from gymnasium.wrappers import RecordVideo

from gymnasium import logger as gymlogger

# в новых версиях вместо set_level используется атрибут min_level
if hasattr(gymlogger, "set_level"):
    gymlogger.set_level(40)
else:
    gymlogger.min_level = 40   # 40 = ERROR
import tensorflow as tf
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
from pprint import pprint
from tqdm import tqdm
import math
import uuid
import glob
import io
import base64
from IPython.display import HTML

from IPython import display as ipythondisplay
import pygame

## Дополнительные функции

Функции для работы с видеозаписями, создает среду Gym с возможностью записи видео и выводит информацию о среде

In [ ]:
def show_video(folder_name):
  mp4list = glob.glob(f'{folder_name}/*.mp4')
  if len(mp4list) > 0:
    mp4 = mp4list[0]
    video = io.open(mp4, 'r+b').read()
    encoded = base64.b64encode(video)
    ipythondisplay.display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
  else:
    print("Could not find video")


def wrap_env(env, folder_name):
  env = RecordVideo(env, folder_name, step_trigger = lambda episode_number: True)
  return env

def create_environment(name):
    folder_name = f"./video/{name}/{uuid.uuid4()}"
    env = wrap_env(gym.make(name, render_mode="rgb_array"), folder_name)
    spec = gym.spec(name)
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {spec.max_episode_steps}")
    print(f"Nondeterministic: {spec.nondeterministic}")
    # Атрибут reward_range удален, так как он не поддерживается RecordVideo в этой версии
    print(f"Reward Threshold: {spec.reward_threshold}")
    return env, folder_name

## Создание агентов

 Позволяет агенту взаимодействовать со средой CliffWalking-v0, выполнять действия, определенные политикой, и записывать видео

In [ ]:
class PolicyIterationAgent:
    '''
    Класс, эмулирующий работу агента
    '''
    def __init__(self, env):
        self.env = env
        self.observation_dim = env.observation_space.n
        self.actions_variants = np.arange(env.action_space.n)
        self.policy_probs = np.full((self.observation_dim, len(self.actions_variants)), 1.0 / len(self.actions_variants))
        self.state_values = np.zeros(shape=(self.observation_dim))
        self.maxNumberOfIterations = 1000
        self.theta=1e-6
        self.gamma=0.99

    def print_policy(self):
        print('Стратегия:')
        pprint(self.policy_probs)

    def policy_evaluation(self):
        for iterations in range(self.maxNumberOfIterations):
            valueFunctionVectorNextIteration = np.zeros(shape=(self.observation_dim))
            for state in range(self.observation_dim):
                action_probabilities = self.policy_probs[state]
                outerSum = 0
                for action, prob in enumerate(action_probabilities):
                    innerSum = 0
                    for probability, next_state, reward, isTerminalState in self.env.P[state][action]:
                        innerSum += probability * (reward + self.gamma * self.state_values[next_state])
                    outerSum += prob * innerSum
                valueFunctionVectorNextIteration[state] = outerSum

            if np.max(np.abs(valueFunctionVectorNextIteration - self.state_values)) < self.theta:
                self.state_values = valueFunctionVectorNextIteration
                break
            self.state_values = valueFunctionVectorNextIteration
        return self.state_values

    def policy_improvement(self):
        improvedPolicy = np.zeros((self.observation_dim, len(self.actions_variants)))
        for state in range(self.observation_dim):
            qvalues = np.zeros(len(self.actions_variants))
            for action in range(len(self.actions_variants)):
                for probability, next_state, reward, isTerminalState in self.env.P[state][action]:
                    qvalues[action] += probability * (reward + self.gamma * self.state_values[next_state])

            bestActionIndex = np.where(qvalues == np.max(qvalues))[0]
            improvedPolicy[state, bestActionIndex] = 1.0 / len(bestActionIndex)
        return improvedPolicy

    def policy_iteration(self, cnt):
        for i in tqdm(range(1, cnt+1)):
            old_policy = self.policy_probs.copy()
            self.state_values = self.policy_evaluation()
            self.policy_probs = self.policy_improvement()
            if np.all(old_policy == self.policy_probs):
                print(f'Алгоритм сошелся на шаге {i}.')
                break

def play_agent(agent):
    # Используем CliffWalking-v0 (v1 может иметь нюансы в зависимости от версии gymnasium)
    name = 'CliffWalking-v0'
    folder_name = f"./video/{name}/{uuid.uuid4()}"
    env = gym.make(name, render_mode="rgb_array")
    env = RecordVideo(env, folder_name, episode_trigger = lambda episode_number: True)

    state, info = env.reset()
    done = False
    while not done:
        p = agent.policy_probs[state]
        action = np.random.choice(len(agent.actions_variants), p=p)
        state, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            done = True

    env.close() # Критически важно для сохранения видео
    return folder_name

Установка виртуального дисплея

In [ ]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()

## Среда обучения

В качестве среды обучения с подкреплением был выбран Toy Text / Cliff Walking из библиотеки Gym.

### Описание

Плата представляет собой матрицу 4x12 с (с использованием индексации матрицы NumPy):

[3, 0] как начало слева внизу

[3, 11] как цель внизу справа

[3, 1..10] в виде скалы внизу в центре.

Если агент наступит на скалу, он вернется к началу. Эпизод заканчивается, когда агент достигает цели.

### Действия
Существует 4 дискретных детерминированных действия:

0: двигаться вверх

1: двигаться вправо

2: двигаться вниз

3: двигаться влево

### Наблюдения
Возможных состояний 3x12 + 1. На самом деле агент не может находиться ни у обрыва, ни у ворот (поскольку это приводит к концу эпизода). Остаются все позиции первых трех строк плюс нижняя левая ячейка. Наблюдение — это просто текущая позиция, закодированная как сглаженный индекс .

### Награда
Каждый временной шаг приносит -1 награду, а шаг в скалу - -100 награды.

## Обучение агента

Обучает агента, который должен научиться двигаться по сетке, избегая обрыва, с помощью алгоритма Policy Iteration

In [ ]:
# Создание среды
env_wrapped = gym.make('CliffWalking-v1')
env = env_wrapped.unwrapped
env.reset()

# Обучение агента
agent = PolicyIterationAgent(env)
agent.print_policy()
agent.policy_iteration(1000)
agent.print_policy()

Стратегия:
array([[0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],

  5%|▌         | 53/1000 [00:00<00:11, 84.61it/s]

Алгоритм сошелся на шаге 54.
Стратегия:
array([[0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25      ],
       [0.25      , 0.25      , 0.25      , 0.25

Запуск обученного агента в среде CliffWalking-v1, демонстрируя его способность успешно пройти по сетке, избегая обрыва,  и сохраняет видеозапись его действий

In [ ]:
import glob
import os
import time

# Запуск агента
folder = play_agent(agent)

# Небольшая пауза для завершения записи видео на диск
time.sleep(2)

try:
    # Ищем все .mp4 файлы в текущей сессии записи
    video_files = glob.glob(f'{folder}/*.mp4')
    # Сортируем по размеру, чтобы найти самый содержательный файл
    video_files.sort(key=os.path.getsize, reverse=True)

    valid_video_found = False
    for video_path in video_files:
        size = os.path.getsize(video_path)
        if size > 2000:  # Игнорируем файлы меньше 2Кб
            print(f"Отображение видео: {video_path} ({size} байт)")
            show_video(folder)
            valid_video_found = True
            break

    if not valid_video_found:
        print("Валидное видео не найдено. Попробуйте запустить обучение еще раз, возможно агент зациклился.")
except Exception as e:
    print(f"Ошибка при поиске видео: {e}")

Action Space: Discrete(4)
Observation Space: Discrete(48)
Max Episode Steps: None
Nondeterministic: False
Reward Threshold: None
Отображение видео: ./video/CliffWalking-v1/b416d49b-6ca0-4cb5-ae7b-f30e5e37b439/rl-video-step-26.mp4 (30906 байт)


## Вывод

В ходе выполнения лабораторной работы я ознакомился с алгоритмом итерации политик, который является одним из ключевых методов обучения с подкреплением. Алгоритм  работает за счет поочередного оценивания и улучшения политики агента.

Оценивание политики: Алгоритм вычисляет значения для каждого состояния в среде, основываясь на текущей политике агента.
Улучшение политики: Используя полученные значения, алгоритм обновляет политику агента, выбирая действия, которые максимизируют ожидаемое вознаграждение.
Этот процесс повторяется до тех пор, пока политика не станет оптимальной, то есть агент будет выбирать действия, которые максимально повышают его шансы на получение максимальной награды.

Код, который я анализировал,  демонстрирует применение этого алгоритма к задаче CliffWalking. В этой задаче агент должен найти путь через сетку, избегая обрыва, который приводит к потере награды. Код успешно обучил агента решать эту задачу, используя алгоритм итерации политик.